# Diabetic retinopathy grading with EfficientNetB3

Classifies Gaussian-filtered retinal (fundus) photographs into the five diabetic retinopathy grades with a transfer-learned EfficientNetB3.

| grade | meaning |
|---|---|
| 0 | No DR |
| 1 | Mild |
| 2 | Moderate |
| 3 | Severe |
| 4 | Proliferative DR |

Data: [APTOS 2019 as 224x224 Gaussian-filtered images](https://www.kaggle.com/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered) on Kaggle (3,662 images, CC0).

**How to run:** `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`. Section 2 needs a Kaggle token and says what to do if it cannot find one.

**What changed since v1** (v1 reached 0.8038 test accuracy on the same test split):

1. *Pixel range.* Keras' EfficientNet rescales and normalizes inside the model and expects 0-255 input. v1 also divided by 255 in `ImageDataGenerator`, so the pixels were scaled down twice. Images now go in as raw 0-255 floats.
2. *Two-phase training.* First the new head is trained with the backbone frozen, then the backbone is unfrozen at a low learning rate with its BatchNorm layers kept frozen. v1 trained everything, BatchNorm included, from the first step (its validation accuracy was 0.05 after epoch 1).
3. *Class weights.* The grades are imbalanced (about half of the images are "No DR"), so the loss is weighted by the square root of the balanced class weights.
4. *Metrics that fit the task.* Quadratic weighted kappa (the standard metric for DR grading), a per-class report, a confusion matrix, and sensitivity / specificity / AUC for "any DR" and "referable DR", both derived from the same softmax output.
5. *Input pipeline.* A `tf.data` pipeline with augmentation layers inside the model replaces the CPU-bound `ImageDataGenerator`, so augmentation runs on the GPU.

## 1. Setup

Every hyperparameter lives in this cell, so the next experiment is a one-line change.

In [ ]:
import os
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers

# ---- configuration ----------------------------------------------------------
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 5
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

HEAD_EPOCHS, HEAD_LR = 10, 1e-3   # phase 1: new head only, backbone frozen
FINE_EPOCHS, FINE_LR = 30, 3e-5   # phase 2: backbone unfrozen, BatchNorm still frozen
USE_CLASS_WEIGHTS = True          # square root of the "balanced" class weights

V1_TEST_ACCURACY, V1_TEST_LOSS = 0.8038, 0.6482  # original notebook, same test split

DATA_DIR = "/content/dr_dataset"    # where the Kaggle download is unzipped
IMAGE_DIR = "/content/all_images"   # every image in one folder (see section 2)
CSV_PATH = os.path.join(DATA_DIR, "train.csv")

keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow {tf.__version__} | Keras {keras.__version__} | GPUs: {[g.name for g in gpus]}")
if not gpus:
    print("WARNING: no GPU found. Use Runtime -> Change runtime type -> GPU, then run again.")

## 2. Get the data from Kaggle

The download needs a Kaggle API token. The cell below looks for one in this order and never prints it:

1. **Environment variables** that are already set (`KAGGLE_API_TOKEN`, or `KAGGLE_USERNAME` + `KAGGLE_KEY`).
2. **Colab Secrets** (the key icon in the left sidebar). Create a token at <https://www.kaggle.com/settings/api> ("Generate New Token"), add it as a secret named `KAGGLE_API_TOKEN` and switch on *Notebook access*. A legacy key works too, as two secrets named `KAGGLE_USERNAME` and `KAGGLE_KEY`.
3. **An uploaded file**: `kaggle.json` (legacy API key) or `access_token`, dropped into `/content` with the Files panel. This is how v1 did it.

The images arrive in one folder per grade. `train.csv` refers to them by id only, so they are gathered into a single folder.

In [ ]:
KAGGLE_DATASET = "sovitrath/diabetic-retinopathy-224x224-gaussian-filtered"


def have_kaggle_env():
    return bool(os.environ.get("KAGGLE_API_TOKEN")) or bool(
        os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")
    )


def setup_kaggle_credentials():
    "Make a Kaggle token visible to the kaggle CLI and say where it came from. Never prints the token."
    if have_kaggle_env():
        return "environment variables"

    try:
        from google.colab import userdata  # Colab Secrets
    except ImportError:
        userdata = None
    if userdata is not None:
        for name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                os.environ[name] = userdata.get(name)
            except Exception:  # secret missing, or notebook access not switched on
                pass
        if have_kaggle_env():
            return "Colab Secrets"

    kaggle_home = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_home, exist_ok=True)
    for filename in ("kaggle.json", "access_token"):
        uploaded = os.path.join("/content", filename)
        target = os.path.join(kaggle_home, filename)
        if os.path.exists(uploaded):
            shutil.copy(uploaded, target)
        if os.path.exists(target):
            os.chmod(target, 0o600)
            return f"{filename} file"

    raise RuntimeError(
        "No Kaggle credentials found. Create a token at https://www.kaggle.com/settings/api, "
        "add it as a Colab secret named KAGGLE_API_TOKEN (key icon in the left sidebar, with "
        "Notebook access switched on) and run this cell again. Alternatively upload "
        "kaggle.json or access_token to /content."
    )


if not os.path.exists(CSV_PATH):
    print("Kaggle credentials from:", setup_kaggle_credentials())
    get_ipython().system("pip install -q -U kaggle")
    get_ipython().system(f"kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_DIR} --unzip")
assert os.path.exists(CSV_PATH), f"{CSV_PATH} is missing: the Kaggle download failed, see its message above."

# train.csv names images by id only, so gather every PNG in one folder
os.makedirs(IMAGE_DIR, exist_ok=True)
moved = 0
for root, _, files in os.walk(DATA_DIR):
    for name in files:
        if name.lower().endswith(".png"):
            shutil.move(os.path.join(root, name), os.path.join(IMAGE_DIR, name))
            moved += 1
print(f"moved {moved} images, {len(os.listdir(IMAGE_DIR))} now in {IMAGE_DIR}")

## 3. Split and class weights

Same seed and stratification as v1, so the test split is the same 367 images (train / validation / test = 2,965 / 330 / 367).

In [ ]:
df = pd.read_csv(CSV_PATH).rename(columns={"id_code": "image", "diagnosis": "level"})
df["image_path"] = df["image"].apply(lambda name: os.path.join(IMAGE_DIR, name + ".png"))
df["level"] = df["level"].astype(int)

missing = int((~df["image_path"].apply(os.path.exists)).sum())
assert missing == 0, f"{missing} images listed in train.csv were not found in {IMAGE_DIR}"

train_val, test = train_test_split(df, test_size=0.1, random_state=SEED, stratify=df["level"])
train, val = train_test_split(train_val, test_size=0.1, random_state=SEED, stratify=train_val["level"])
train, val, test = (frame.reset_index(drop=True) for frame in (train, val, test))

splits = {"train": train, "val": val, "test": test}
distribution = pd.DataFrame({name: frame["level"].value_counts().sort_index() for name, frame in splits.items()})
distribution.index = CLASS_NAMES
print(distribution)

if USE_CLASS_WEIGHTS:
    balanced = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train["level"].to_numpy())
    class_weight = {grade: float(np.sqrt(weight)) for grade, weight in enumerate(balanced)}
else:
    class_weight = None
print("class weights:", class_weight)

## 4. Input pipeline

Pixels stay in the **0-255** range on purpose: the EfficientNet model rescales and normalizes them itself. The cell prints the pixel range of one batch as a check.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    image = tf.io.decode_png(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, IMG_SIZE)  # float32, values stay in 0-255
    return image, label


def make_dataset(frame, training):
    ds = tf.data.Dataset.from_tensor_slices(
        (frame["image_path"].tolist(), frame["level"].to_numpy(dtype="int32"))
    )
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_dataset(train, training=True)
val_ds = make_dataset(val, training=False)
test_ds = make_dataset(test, training=False)

# Sanity check: EfficientNet must get raw 0-255 pixels (v1 fed it 0-1 values).
images, labels = next(iter(train_ds))
pixel_min, pixel_max = float(tf.reduce_min(images)), float(tf.reduce_max(images))
print(f"batch {tuple(images.shape)} {images.dtype.name} | pixel range {pixel_min:.0f} - {pixel_max:.0f}")
assert pixel_max > 1.0, "pixels look rescaled to 0-1; EfficientNet expects 0-255"

fig, axes = plt.subplots(2, 4, figsize=(12, 6.5))
for ax, image, label in zip(axes.ravel(), images.numpy(), labels.numpy()):
    ax.imshow(image.astype("uint8"))
    ax.set_title(f"grade {label}: {CLASS_NAMES[label]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Model

EfficientNetB3 with ImageNet weights, then global average pooling and a small dense head (the same head as v1). The augmentation layers sit at the start of the model and are only active while training. The backbone is called with `training=False`, so its BatchNorm layers always run in inference mode, which keeps the pretrained statistics intact.

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.1, fill_mode="nearest"),  # +-36 degrees
        layers.RandomTranslation(0.1, 0.1, fill_mode="nearest"),
        layers.RandomZoom(0.2, fill_mode="nearest"),
    ],
    name="augmentation",
)


def build_model():
    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = data_augmentation(inputs)  # only active while training
    backbone = keras.applications.EfficientNetB3(
        include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3)
    )
    backbone.trainable = False
    x = backbone(x, training=False)  # BatchNorm always runs in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="dr_efficientnetb3"), backbone


def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=False,  # the random augmentation layers cannot be XLA-compiled
    )


model, backbone = build_model()
model.summary()

## 6. Train

**Phase 1** trains only the new head while the backbone is frozen, so the randomly initialised head cannot damage the pretrained features. **Phase 2** unfreezes the backbone and fine-tunes everything except BatchNorm with a much smaller learning rate. Both phases stop early on validation loss and restore the best weights.

In [ ]:
# Phase 1: new head only
compile_model(model, HEAD_LR)
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    class_weight=class_weight,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
)

In [ ]:
# Phase 2: unfreeze the backbone but keep BatchNorm frozen, then fine-tune gently
backbone.trainable = True
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

compile_model(model, FINE_LR)  # recompile so the new trainable flags take effect
start_epoch = len(history_head.epoch)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=start_epoch,
    epochs=start_epoch + FINE_EPOCHS,
    class_weight=class_weight,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
    ],
)

## 7. Training curves

The dashed line marks where the backbone was unfrozen.

In [ ]:
def joined(key):
    return history_head.history[key] + history_fine.history[key]


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, key, title in zip(axes, ("loss", "accuracy"), ("Loss", "Accuracy")):
    ax.plot(joined(key), label="train")
    ax.plot(joined("val_" + key), label="validation")
    ax.axvline(len(history_head.epoch) - 0.5, color="gray", linestyle="--", label="backbone unfrozen")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

## 8. Evaluate on the held-out test split

Accuracy alone hides how the model errs on an imbalanced, ordered label. **Quadratic weighted kappa** punishes a two-grade miss far more than a one-grade miss, which is why it is the headline metric for DR grading. The binary readouts reuse the same predictions: `any DR` is grade >= 1 and `referable DR` is grade >= 2, both scored with the probability mass the model puts on those grades.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)
y_true = test["level"].to_numpy()

qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
print(f"v2  test accuracy {test_accuracy:.4f} | test loss {test_loss:.4f} | quadratic weighted kappa {qwk:.4f}")
print(f"v1  test accuracy {V1_TEST_ACCURACY:.4f} | test loss {V1_TEST_LOSS:.4f} | (kappa was not measured)")
print()
print(classification_report(
    y_true, y_pred, labels=list(range(NUM_CLASSES)), target_names=CLASS_NAMES, digits=3, zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
ax.set_title("Test confusion matrix (rows: true grade, columns: predicted)")
plt.tight_layout()
plt.show()


def binary_readout(name, truth, score, threshold=0.5):
    predicted = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(truth, predicted, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else float("nan")
    specificity = tn / (tn + fp) if (tn + fp) else float("nan")
    auc = roc_auc_score(truth, score)
    print(f"{name:<26} AUC {auc:.3f} | sensitivity {sensitivity:.3f} | specificity {specificity:.3f}")


binary_readout("any DR (grade >= 1)", (y_true >= 1).astype(int), 1.0 - y_prob[:, 0])
binary_readout("referable DR (grade >= 2)", (y_true >= 2).astype(int), y_prob[:, 2:].sum(axis=1))

## 9. Save the model

The file lands in Colab's temporary storage and disappears with the runtime. Download it from the Files panel on the left, or copy it to Google Drive.

In [ ]:
model.save("dr_efficientnetb3.keras")
size_mb = os.path.getsize("dr_efficientnetb3.keras") / 1e6
print(f"saved dr_efficientnetb3.keras ({size_mb:.0f} MB)")

## Where to go next

- Treat the grade as ordinal: regress on it and tune the cut points on the validation split, which often beats a 5-way softmax on kappa.
- Grad-CAM overlays, to check that the model looks at lesions rather than at image borders or exposure artefacts.
- Test-time augmentation, and several seeds so a result is not a single lucky split.
- A small demo that takes a fundus photo and returns the grade with its probabilities.